# Notebook 3b - CNNs on the microstructures

**UKACM Autumn School: AI for Computational Mechanics**

### Where Notebook 3a left off

Notebook 3a built a convolutional neural network from its parts, on small examples. This notebook
trains one on the real data and asks two questions: does it work, and does it do better than the
methods from Notebooks 1 and 2?

### The data, in one paragraph

1485 images of the cross-section of a unidirectional fibre composite, each a periodic unit cell, 64x64
pixels, 1 for fibre and 0 for matrix. For each one, a finite element model gave the transverse
stiffness in the two in-plane directions, $E_{22}$ and $E_{33}$. Two targets are used throughout:

$$E_{mean} = \tfrac12\left(E_{22} + E_{33}\right) \qquad \text{(how stiff, on average)}$$

$$\Delta E = E_{22} - E_{33} \qquad \text{(how different the two directions are: the anisotropy)}$$

Notebook 1 showed that the fibre volume fraction predicts $E_{mean}$ well and tells you nothing at all
about $\Delta E$. Notebook 2 showed that a set of hand-chosen geometric descriptors recovers most of
$\Delta E$.

### What you will do

| Part | Question |
|---|---|
| 1 | What do the two targets look like? |
| 2 | Can a CNN predict the average stiffness? |
| 3 | Can it predict the anisotropy, and does it beat the descriptors? |
| 4 | Does the CNN really use the arrangement of the fibres? (the shuffle test) |
| 5 | How far does a unit need to see? |
| 6 | Every method side by side |
| 7 | What did the network learn? |

Several cells train networks. Each prints its own run time. The longest takes a few minutes on a
free Colab CPU.

In [ ]:
# --- Setup -------------------------------------------------------------------
# Run this first.

import os, io, time, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import Rectangle
from IPython.display import HTML, display
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Text, Checkbox

import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import ndimage
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)          # matches a free Colab CPU runtime

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10, "axes.grid": True,
    "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False,
    "animation.embed_limit": 60,
})

C_DATA, C_FIT, C_ALT, C_BAD = "#3B6EA5", "#C25E00", "#4C9A5E", "#A8323E"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("numpy", np.__version__, "| torch", torch.__version__, "| device", DEVICE)
if DEVICE == "cpu":
    print("No GPU found. Every timing quoted below is for 2 CPU cores.")


### Loading the data

Four files, the same as in Notebook 2:

- `microstructure_labels.csv`, one row per microstructure, with the stiffness values
- `microstructures_64.npz`, the images
- `microstructure_descriptors_core14_64.npz`, the 14 named descriptors of Notebook 2
- `microstructure_descriptors_v17_30_64.npz`, a larger research set of 30 descriptors

The descriptors were computed from these same 64x64 images, so every comparison below uses
the same information in different forms. 15 rows flagged as outliers are dropped.

The following cell downloads the shared CNN dataset automatically.


In [ ]:
# --- Load data ---------------------------------------------------------------
DATA_URL = "https://raw.githubusercontent.com/CEMS-Lab/autumn-school/main/datasets/machine_learning/NB3_data.zip"  # shared CNN dataset
DATA_DIR = "."

FILES = ["microstructure_labels.csv", "microstructures_64.npz",
         "microstructure_descriptors_core14_64.npz",
         "microstructure_descriptors_v17_30_64.npz"]

def _have():
    return all(os.path.exists(os.path.join(DATA_DIR, f)) for f in FILES)

if not _have() and DATA_URL:
    print("Downloading ...")
    with urllib.request.urlopen(DATA_URL) as r:
        zipfile.ZipFile(io.BytesIO(r.read())).extractall(DATA_DIR)

if not _have():
    try:
        from google.colab import files
        print("Select:", ", ".join(FILES))
        files.upload()
    except ImportError:
        raise FileNotFoundError(
            "Put " + ", ".join(FILES) + " beside the notebook, or set DATA_URL above.")

df = pd.read_csv(os.path.join(DATA_DIR, "microstructure_labels.csv"))
df["E_mean"] = (df.E22 + df.E33) / 2
df["dE"]     =  df.E22 - df.E33
df["vf"]     =  df.vol_frac / 100.0

clean = df[~df.outlier_flag].copy().reset_index(drop=True)

_img = np.load(os.path.join(DATA_DIR, "microstructures_64.npz"), allow_pickle=True)
IMAGES, IMG_KEYS = _img["images"], list(_img["keys"])
IMG_IDX = {k: i for i, k in enumerate(IMG_KEYS)}

_dsc = np.load(os.path.join(DATA_DIR, "microstructure_descriptors_core14_64.npz"),
               allow_pickle=True)
DESC_ALL, DESC_KEYS = _dsc["D"], list(_dsc["keys"])
DESC_NAMES    = [str(v) for v in _dsc["names"]]
DESC_MEANINGS = [str(v) for v in _dsc["meanings"]]
DESC_IDX = {k: i for i, k in enumerate(DESC_KEYS)}

_d30 = np.load(os.path.join(DATA_DIR, "microstructure_descriptors_v17_30_64.npz"),
               allow_pickle=True)
D30_ALL, D30_KEYS = _d30["D"], list(_d30["keys"])
D30_NAMES = [str(v) for v in _d30["names"]]
D30_IDX = {k: i for i, k in enumerate(D30_KEYS)}

X_IMG  = IMAGES[[IMG_IDX[k]  for k in clean.key]].astype(np.float32)
X_DESC = DESC_ALL[[DESC_IDX[k] for k in clean.key]].astype(np.float32)
X_D30  = D30_ALL[[D30_IDX[k]  for k in clean.key]].astype(np.float32)
y_mean = clean["E_mean"].values.astype(np.float32)
y_dE   = clean["dE"].values.astype(np.float32)

print(f"{len(clean)} microstructures kept, {int(df.outlier_flag.sum())} outliers dropped")
print(f"images      {X_IMG.shape}   values in {{{X_IMG.min():.0f}, {X_IMG.max():.0f}}}")
print(f"descriptors {X_DESC.shape} (named)   {X_D30.shape} (full bank)")
print()
print("the 14 named descriptors Notebook 2 teaches")
for _n, _m in zip(DESC_NAMES, DESC_MEANINGS):
    print(f"   {_n:20s} {_m}")


---

# Part 1 - The two targets

The first figure shows why the two targets are different problems.

In [ ]:
# --- The two targets against volume fraction -----------------------------------
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
jit = np.random.default_rng(0).normal(0, 0.006, len(clean))
a1.scatter(clean.vf + jit, y_mean, s=5, alpha=0.4, color=C_DATA)
a1.set_xlabel("fibre volume fraction $V_f$"); a1.set_ylabel("$E_{mean}$ (GPa)")
a1.set_title("average stiffness rises steadily with $V_f$", fontsize=10)
a2.scatter(clean.vf + jit, y_dE, s=5, alpha=0.4, color=C_FIT)
a2.axhline(0, color="k", lw=0.8)
a2.set_xlabel("fibre volume fraction $V_f$"); a2.set_ylabel("$\\Delta E$ (GPa)")
a2.set_title("anisotropy scatters around zero at every $V_f$", fontsize=10)
plt.tight_layout(); plt.show()

hi, lo = int(np.argmax(y_dE)), int(np.argmin(y_dE))
same_vf = clean.vf.values
fig, axes = plt.subplots(1, 2, figsize=(7, 3.6))
for ax, i in zip(axes, [hi, lo]):
    ax.imshow(X_IMG[i], cmap="gray", interpolation="nearest")
    ax.set_title(f"$V_f$ = {same_vf[i]:.2f},  $\\Delta E$ = {y_dE[i]:+.2f} GPa", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.suptitle("the most positive and the most negative anisotropy in the data", y=1.02, fontsize=10)
plt.tight_layout(); plt.show()


**What the figures show.** Left: every group of cells with the same volume fraction sits at about
the same $E_{mean}$, so the amount of fibre is almost all you need. Right: at every volume fraction
$\Delta E$ spreads above and below zero, so the amount of fibre tells you nothing about it.
Anisotropy comes from **how the fibres are arranged**: whether they happen to line up or cluster more
in one direction than the other. Bottom: the two most extreme cells. Try to see the difference by
eye. It is subtle, which is what makes this a hard target.

A CNN reads the arrangement directly from the image. The rest of this notebook tests whether it
learns to use it.

---

# Part 2 - A CNN for the average stiffness

### The network

The network is the one drawn in Notebook 3a, Part 3.6: three blocks of 3x3 convolution, ReLU and
2x2 max pooling, with circular padding because the cells are periodic, then a small dense network.

In [ ]:
# --- The network --------------------------------------------------------------
def make_cnn(widths=(16, 32, 64), head=64, in_size=64, pad_mode="circular", seed=SEED):
    torch.manual_seed(seed)
    layers, c = [], 1
    for w in widths:
        layers += [nn.Conv2d(c, w, 3, padding=1, padding_mode=pad_mode),
                   nn.ReLU(), nn.MaxPool2d(2)]
        c = w
    s = in_size // (2 ** len(widths))
    layers += [nn.Flatten(), nn.Linear(c * s * s, head), nn.ReLU(), nn.Linear(head, 1)]
    return nn.Sequential(*layers)

cnn_demo = make_cnn()
N_PARAM_CNN = sum(p.numel() for p in cnn_demo.parameters())

conv_par = sum(p.numel() for m in cnn_demo if isinstance(m, nn.Conv2d) for p in m.parameters())
dense_par = N_PARAM_CNN - conv_par

# the Notebook 2 flattened MLP, for comparison
mlp_flat_sizes = [4096, 256, 64, 1]
N_PARAM_MLP = sum(a * b + b for a, b in zip(mlp_flat_sizes[:-1], mlp_flat_sizes[1:]))

print(cnn_demo)
print()
print(f"CNN total parameters      {N_PARAM_CNN:>10,d}")
print(f"  in the convolutions     {conv_par:>10,d}   ({100*conv_par/N_PARAM_CNN:.1f}%)")
print(f"  in the dense head       {dense_par:>10,d}")
print(f"NB2 flattened MLP         {N_PARAM_MLP:>10,d}")


Most of the weights are in the dense part, not in the convolutions that do the pattern finding.
This is because the convolutions share their weights.

### The training budget

The number of passes through the training data, called **epochs**, is set once here. If a GPU is
available the counts are doubled, since the same run then costs a fraction of the time.

In [ ]:
# --- Budget -------------------------------------------------------------------
EPOCHS_MEAN = 20 if DEVICE == "cpu" else 40     # target E_mean, Part 2
EPOCHS_DE   = 40 if DEVICE == "cpu" else 80     # target dE, Parts 3 and 4
EPOCHS_RF   = 20 if DEVICE == "cpu" else 40     # each of the four networks in Part 5
print(f"E_mean: {EPOCHS_MEAN} epochs   dE: {EPOCHS_DE} epochs   Part 5: {EPOCHS_RF} epochs each, on {DEVICE}")


### More training images from the physics: augmentation

The training set has about 1100 images. **Augmentation** makes more by transforming each image in a
way that does not change its answer. The transformations must come from the physics.

- **Periodic shift.** Moving a periodic cell cyclically in $x$ or $y$ gives the same material with a
  different origin. Stiffness is unchanged. This is exact.
- **Mirror in $x$ or in $y$.** A mirrored cell has the same stiffness in each direction, so both
  $E_{mean}$ and $\Delta E$ are unchanged.
- **Rotation by 90 degrees.** This one swaps the two directions, so $E_{22}$ and $E_{33}$ swap and
  $\Delta E$ changes sign. It could be used, but only by flipping the sign of the target too. It is
  left out here.

A transformation that does change the answer would teach the network a wrong relationship.

In [ ]:
# --- Augmentation -------------------------------------------------------------
def augment(xb, rng=np.random):
    sy, sx = rng.randint(0, xb.shape[-2]), rng.randint(0, xb.shape[-1])
    xb = torch.roll(xb, shifts=(int(sy), int(sx)), dims=(2, 3))    # periodic roll
    if rng.rand() < 0.5: xb = torch.flip(xb, dims=[3])             # mirror in x
    if rng.rand() < 0.5: xb = torch.flip(xb, dims=[2])             # mirror in y
    return xb

np.random.seed(SEED)
ex = torch.tensor(X_IMG[3]).view(1, 1, 64, 64)
fig, axes = plt.subplots(1, 6, figsize=(14, 2.7))
axes[0].imshow(ex[0, 0], cmap="gray", interpolation="nearest")
axes[0].set_title("original", fontsize=9)
for k in range(1, 6):
    a = augment(ex.clone())
    axes[k].imshow(a[0, 0], cmap="gray", interpolation="nearest")
    axes[k].set_title(f"augmented {k}\n$V_f$ = {a.mean():.3f}", fontsize=9)
for a in axes:
    a.set_xticks([]); a.set_yticks([]); a.grid(False)
plt.suptitle("periodic roll plus flips: the same cell, a different origin", y=1.06, fontsize=10)
plt.tight_layout(); plt.show()


### Training

One helper function trains every CNN in this notebook. It uses the Adam optimiser, a
refinement of the gradient descent of Notebook 3a, and the mean squared error loss. The target is
rescaled to zero mean and unit spread using the training set only, and the test $R^2$ is recorded
every two epochs so it can be plotted.

In [ ]:
# --- Training helper ----------------------------------------------------------
def train_cnn(model, Xtr, ytr, Xte, yte, epochs=40, bs=32, lr=1e-3,
              augmented=True, eval_every=2, seed=SEED, tag=""):
    rng = np.random.RandomState(seed)
    torch.manual_seed(seed)
    model = model.to(DEVICE)

    mu, sd = float(ytr.mean()), float(ytr.std())          # target standardisation, train only
    A = torch.tensor(Xtr).unsqueeze(1).to(DEVICE)
    B = torch.tensor((ytr - mu) / sd).view(-1, 1).to(DEVICE)
    P = torch.tensor(Xte).unsqueeze(1).to(DEVICE)

    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.MSELoss()
    n = len(A)
    hist_ep, hist_r2, hist_loss = [], [], []

    t0 = time.time()
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n)
        run = 0.0
        for i in range(0, n, bs):
            j  = perm[i:i + bs]
            xb = augment(A[j].clone(), rng) if augmented else A[j]
            opt.zero_grad()
            l = lossf(model(xb), B[j])
            l.backward(); opt.step()
            run += l.item() * len(j)
        hist_loss.append(run / n)
        if ep % eval_every == eval_every - 1 or ep == epochs - 1:
            model.eval()
            with torch.no_grad():
                pr = model(P).cpu().numpy().ravel() * sd + mu
            hist_ep.append(ep + 1); hist_r2.append(r2_score(yte, pr))
    elapsed = time.time() - t0

    model.eval()
    with torch.no_grad():
        pred = model(P).cpu().numpy().ravel() * sd + mu
    r2 = r2_score(yte, pred)
    print(f"{tag}{epochs} epochs, augmented={augmented}: {elapsed:.1f} s "
          f"({elapsed/epochs:.2f} s per epoch), test R2 = {r2:.4f}")
    return dict(r2=r2, elapsed=elapsed, pred=pred, ep=hist_ep, r2_hist=hist_r2,
                loss=hist_loss, model=model)


In [ ]:
# --- Train the CNN on E_mean --------------------------------------------------
idx = np.arange(len(clean))
itr, ite = train_test_split(idx, test_size=0.25, random_state=SEED)
Xtr_img, Xte_img = X_IMG[itr], X_IMG[ite]
ym_tr, ym_te = y_mean[itr], y_mean[ite]
yd_tr, yd_te = y_dE[itr],  y_dE[ite]

print(f"train {len(itr)}   test {len(ite)}")
res_mean = train_cnn(make_cnn(), Xtr_img, ym_tr, Xte_img, ym_te,
                     epochs=EPOCHS_MEAN, tag="CNN on E_mean, ")
R2_CNN_MEAN = res_mean["r2"]


In [ ]:
# --- How it trained -----------------------------------------------------------
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.plot(res_mean["loss"], color=C_DATA, lw=1.8)
a1.set_yscale("log"); a1.set_xlabel("epoch"); a1.set_ylabel("training MSE (standardised target)")
a1.set_title("training loss", fontsize=10)

a2.scatter(ym_te, res_mean["pred"], s=12, alpha=0.5, color=C_DATA)
lim = [ym_te.min() - 0.4, ym_te.max() + 0.4]
a2.plot(lim, lim, "k--", lw=1)
a2.set_xlim(lim); a2.set_ylim(lim)
a2.set_xlabel("true $E_{mean}$ (GPa)"); a2.set_ylabel("predicted $E_{mean}$ (GPa)")
a2.set_title(f"CNN on the raw image, test $R^2$ = {R2_CNN_MEAN:.4f}", fontsize=10)
plt.tight_layout(); plt.show()


**What the two panels show.** Left: the training loss falls steadily. Right: each dot is a test
cell that the network never saw, its true $E_{mean}$ against the prediction. The dots lie close to
the dashed diagonal, so the CNN predicts the average stiffness well.

### Comparison with simpler models

Two baselines on the same split: the straight line in volume fraction from Notebook 1, and the
flattened dense network from Notebook 2.

In [ ]:
# --- Notebook 1 and Notebook 2 baselines, recomputed on this split -----------
vf_all = clean["vf"].values.reshape(-1, 1)
lin_mean = LinearRegression().fit(vf_all[itr], ym_tr)
R2_LIN_MEAN = r2_score(ym_te, lin_mean.predict(vf_all[ite]))

X_FLAT = X_IMG.reshape(len(clean), -1)

def train_flat_mlp(Xtr_, ytr_, Xte_, yte_, epochs=12, bs=32, lr=1e-3, seed=SEED, tag=""):
    torch.manual_seed(seed)
    m = nn.Sequential(nn.Linear(4096, 256), nn.ReLU(),
                      nn.Linear(256, 64), nn.ReLU(), nn.Linear(64, 1))
    mu, sd = float(ytr_.mean()), float(ytr_.std())
    A = torch.tensor(Xtr_); B = torch.tensor((ytr_ - mu) / sd).view(-1, 1)
    P = torch.tensor(Xte_)
    opt = torch.optim.Adam(m.parameters(), lr=lr); lf = nn.MSELoss()
    t0 = time.time(); n = len(A)
    for _ in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, bs):
            j = perm[i:i + bs]
            opt.zero_grad(); lf(m(A[j]), B[j]).backward(); opt.step()
    el = time.time() - t0
    with torch.no_grad():
        pred = m(P).numpy().ravel() * sd + mu
    r2 = r2_score(yte_, pred)
    print(f"{tag}flattened MLP, {epochs} epochs: {el:.1f} s, test R2 = {r2:.4f}")
    return r2, el

R2_MLP_MEAN, el_mlp_mean = train_flat_mlp(X_FLAT[itr], ym_tr, X_FLAT[ite], ym_te,
                                          tag="E_mean, ")
print(f"\nlinear in Vf only: test R2 = {R2_LIN_MEAN:.4f}")


The straight line in volume fraction already gets most of the way to the CNN's score, and the
flattened dense network does worse than both. For $E_{mean}$ the image adds little, because the
answer depends mainly on how much fibre there is. A fairer test of the CNN needs a target where the
arrangement matters.

---

# Part 3 - The anisotropy

Now train the same network, the same way, on $\Delta E$. The cell after it fits the descriptor models
of Notebook 2 on the same split, so the scores can be compared directly.

In [ ]:
# --- Train the CNN on the anisotropy -----------------------------------------
res_dE = train_cnn(make_cnn(), Xtr_img, yd_tr, Xte_img, yd_te,
                   epochs=EPOCHS_DE, tag="CNN on dE, ")
R2_CNN_DE = res_dE["r2"]


In [ ]:
# --- The descriptor baselines, on the same split -----------------------------
# 14 named descriptors, the baseline Notebook 2 teaches
t0 = time.time()
gb_dE = HistGradientBoostingRegressor(random_state=SEED, max_iter=400).fit(X_DESC[itr], yd_tr)
R2_DESC_DE = r2_score(yd_te, gb_dE.predict(X_DESC[ite]))
el_desc = time.time() - t0
gb_mean = HistGradientBoostingRegressor(random_state=SEED, max_iter=400).fit(X_DESC[itr], ym_tr)
R2_DESC_MEAN = r2_score(ym_te, gb_mean.predict(X_DESC[ite]))

# the full research bank of 30, same model, same split
t0 = time.time()
gb30_dE = HistGradientBoostingRegressor(random_state=SEED, max_iter=400).fit(X_D30[itr], yd_tr)
R2_D30_DE = r2_score(yd_te, gb30_dE.predict(X_D30[ite]))
el_d30 = time.time() - t0
gb30_mean = HistGradientBoostingRegressor(random_state=SEED, max_iter=400).fit(X_D30[itr], ym_tr)
R2_D30_MEAN = r2_score(ym_te, gb30_mean.predict(X_D30[ite]))

lin_dE = LinearRegression().fit(vf_all[itr], yd_tr)
R2_LIN_DE = r2_score(yd_te, lin_dE.predict(vf_all[ite]))

R2_MLP_DE, _ = train_flat_mlp(X_FLAT[itr], yd_tr, X_FLAT[ite], yd_te, tag="dE, ")

print(f"\n{'representation':34s} {'dim':>4s} {'fit s':>7s} {'R2 on dE':>9s}")
print(f"{'14 named descriptors':34s} {X_DESC.shape[1]:4d} {el_desc:7.1f} {R2_DESC_DE:9.4f}")
print(f"{'full bank, 30 descriptors':34s} {X_D30.shape[1]:4d} {el_d30:7.1f} {R2_D30_DE:9.4f}")
print(f"{'volume fraction only, linear':34s} {1:4d} {0.0:7.1f} {R2_LIN_DE:9.4f}")
print(f"{'raw image 64x64, CNN':34s} {64*64:4d} {res_dE['elapsed']:7.1f} {R2_CNN_DE:9.4f}")

# which of the 14 carries the anisotropy
from sklearn.inspection import permutation_importance
pi = permutation_importance(gb_dE, X_DESC[ite], yd_te, n_repeats=5, random_state=SEED)
ord_pi = np.argsort(pi.importances_mean)[::-1]
TOP_DESC = DESC_NAMES[ord_pi[0]]
LAG_TOP  = int(TOP_DESC.split("_")[-1])
print("\npermutation importance on the 14 named descriptors, target dE")
for k in ord_pi[:4]:
    print(f"   {DESC_NAMES[k]:20s} {pi.importances_mean[k]:7.4f}   {DESC_MEANINGS[k]}")
print(f"\ndominant descriptor {TOP_DESC}, a directional statistic at a lag of {LAG_TOP} pixels")

R2_PUB_DE = 0.942      # published tuned directional CNN on this data, quoted for reference only
print(f"published tuned research CNN on dE, for reference only: {R2_PUB_DE:.3f}")


In [ ]:
# --- Parity plot and the test curve ------------------------------------------
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))

a1.plot(res_dE["ep"], res_dE["r2_hist"], "-o", ms=3, color=C_DATA, label="CNN, test $R^2$")
a1.axhline(R2_DESC_DE, color=C_ALT, ls="--", lw=1.8,
           label=f"14 named descriptors, {R2_DESC_DE:.3f}")
a1.axhline(R2_D30_DE, color=C_ALT, ls="-.", lw=1.4,
           label=f"full bank of 30, {R2_D30_DE:.3f}")
a1.axhline(R2_PUB_DE, color=C_FIT, ls=":", lw=1.8,
           label=f"tuned research CNN, published, {R2_PUB_DE:.3f}")
a1.set_xlabel("epoch"); a1.set_ylabel("test $R^2$ on $\\Delta E$")
a1.set_ylim(-0.1, 1.0); a1.legend(fontsize=8, loc="lower right")
a1.set_title("the CNN against its competition", fontsize=10)

a2.scatter(yd_te, res_dE["pred"], s=12, alpha=0.5, color=C_DATA)
lim = [yd_te.min() - 0.2, yd_te.max() + 0.2]
a2.plot(lim, lim, "k--", lw=1)
a2.set_xlim(lim); a2.set_ylim(lim)
a2.set_xlabel("true $\\Delta E$ (GPa)"); a2.set_ylabel("predicted $\\Delta E$ (GPa)")
a2.set_title(f"CNN on $\\Delta E$, test $R^2$ = {R2_CNN_DE:.4f}", fontsize=10)
plt.tight_layout(); plt.show()


**What the two panels show.** Left: the blue curve is the CNN's test $R^2$ on $\Delta E$ as
training goes on. The green lines are gradient boosting on the 14 named descriptors (dashed) and on
the 30 research descriptors (dash-dot). The dotted line is a published CNN on this same data,
designed for the task and trained for much longer, quoted only for scale. Right: predicted against
true $\Delta E$ for the CNN.

**The CNN loses to the descriptors.** Read the printed scores. The CNN saw the whole image. The
descriptors were computed from those same images, so the CNN had at least as much information. It
has about a quarter of a million weights and the longest training run so far. It still scores below
a model fitted to fourteen numbers chosen by a person who understands the mechanics, and that model
took a fraction of the time to fit.

The parity plot shows the CNN has learned something real: the cloud leans along the diagonal. It is
also wide, so much of the anisotropy is still unexplained.


### Try it: compare the models test cell by test cell

Choose a model to see its predictions on the test set, and pick a range of true $\Delta E$ to
highlight. Look at where each model goes wrong: in the middle, where the anisotropy is small, or at
the extremes.

In [ ]:
# --- Interactive: parity plots for the dE models --------------------------------
from ipywidgets import FloatRangeSlider
PRED_DE = {"CNN on the image": res_dE["pred"],
           "14 named descriptors": gb_dE.predict(X_DESC[ite]),
           "30 research descriptors": gb30_dE.predict(X_D30[ite])}

def parity_explorer(model="CNN on the image", highlight=(-2.0, -0.8)):
    p_ = PRED_DE[model]
    sel = (yd_te >= highlight[0]) & (yd_te <= highlight[1])
    fig, ax = plt.subplots(figsize=(5.4, 5))
    ax.scatter(yd_te, p_, s=12, alpha=0.35, color="0.6")
    ax.scatter(yd_te[sel], p_[sel], s=18, color=C_FIT)
    lim = [yd_te.min() - 0.2, yd_te.max() + 0.2]
    ax.plot(lim, lim, "k--", lw=1); ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("true $\\Delta E$ (GPa)"); ax.set_ylabel("predicted $\\Delta E$ (GPa)")
    err_all = np.sqrt(np.mean((p_ - yd_te) ** 2))
    err_sel = np.sqrt(np.mean((p_[sel] - yd_te[sel]) ** 2)) if sel.any() else float("nan")
    ax.set_title(f"{model}: R2 = {r2_score(yd_te, p_):.3f}\n"
                 f"RMS error {err_all:.3f} GPa overall, {err_sel:.3f} GPa in the orange range "
                 f"({int(sel.sum())} cells)", fontsize=9)
    plt.tight_layout(); plt.show()

interact(parity_explorer, model=Dropdown(options=list(PRED_DE.keys())),
         highlight=FloatRangeSlider(value=(-2.0, -0.8), min=-2.0, max=2.0, step=0.1,
                                    continuous_update=False, description="true dE"));


### Why the descriptors win here

- **The descriptors already contain the answer.** The printed importance ranking puts one
  descriptor far above the rest. It compares how fibre pixels are correlated along $x$ and along
  $y$, at the lag printed above. That is close to a direct measure of directional arrangement. The CNN
  has to build something similar out of stacked 3x3 kernels, from about 1100 examples.
- **The signal is small.** $\Delta E$ is the difference of two nearly equal numbers and spans only a
  few GPa around zero.
- **The network is generic.** Nothing in it knows the target compares two directions. The
  published network that reaches the dotted line was designed around that fact.

This does not make CNNs poor tools. It shows that a quick, general CNN is not automatically better
than features chosen with physical insight, and that a baseline like the descriptor model should
always be run.

---

# Part 4 - The shuffle test

Notebook 3a predicted what should happen if every image has its pixels reordered by one fixed
permutation: the fibre fraction survives, the arrangement does not. So a target that depends on the
amount of fibre should survive, and a target that depends on the arrangement should not, **for a
network that relies on neighbouring pixels**. A dense network should not care either way.

The figure shows the shuffled images and checks the first claim with no training at all.

In [ ]:
# --- The same fixed permutation as Notebook 2 --------------------------------
perm_pix = np.random.default_rng(1).permutation(4096)
X_SHUF = X_FLAT[:, perm_pix].reshape(-1, 64, 64)

fig, axes = plt.subplots(1, 4, figsize=(12, 3.3))
for k, i in enumerate([0, 5]):
    axes[2*k].imshow(X_IMG[i], cmap="gray", interpolation="nearest")
    axes[2*k].set_title(f"microstructure\n$V_f$ = {X_IMG[i].mean():.3f}", fontsize=9)
    axes[2*k+1].imshow(X_SHUF[i], cmap="gray", interpolation="nearest")
    axes[2*k+1].set_title(f"pixels shuffled\n$V_f$ = {X_SHUF[i].mean():.3f}", fontsize=9)
for a in axes:
    a.set_xticks([]); a.set_yticks([]); a.grid(False)
plt.suptitle("one fixed permutation, applied to every image in the dataset", y=1.05)
plt.tight_layout(); plt.show()

vf_real = X_IMG.mean(axis=(1, 2)); vf_shuf = X_SHUF.mean(axis=(1, 2))
print(f"largest change in fibre fraction caused by shuffling, over all images: {np.abs(vf_real - vf_shuf).max():.1e}")
for lab_, v_ in [("real", vf_real), ("shuffled", vf_shuf)]:
    r2_ = r2_score(ym_te, LinearRegression().fit(v_[itr, None], ym_tr).predict(v_[ite, None]))
    print(f"E_mean from the pixel fibre fraction of the {lab_:8s} images, straight line: test R2 = {r2_:.4f}")


The fibre fraction is unchanged by shuffling, so a straight line in the fibre fraction measured from
the pixels gives exactly the same score on the shuffled images as on the real ones. Shuffling
therefore cannot hurt $E_{mean}$ much for any model that can count fibre pixels, and the test only
tells us something on $\Delta E$.

Now train both networks on the shuffled images, with $\Delta E$ as the target. The CNN run is the
same length as in Part 3.

In [ ]:
# --- The shuffle test on dE: dense network and CNN -------------------------------
R2_MLP_DE_SHUF, _ = train_flat_mlp(X_FLAT[:, perm_pix][itr], yd_tr, X_FLAT[:, perm_pix][ite], yd_te,
                                   tag="dE, shuffled pixels, ")
res_dE_shuf = train_cnn(make_cnn(), X_SHUF[itr], yd_tr, X_SHUF[ite], yd_te,
                        epochs=EPOCHS_DE, tag="CNN on dE, shuffled pixels, ")
R2_CNN_DE_SHUF = res_dE_shuf["r2"]

print()
print(f"test R2 on dE          {'real images':>12s} {'shuffled':>10s}")
print(f"flattened dense net    {R2_MLP_DE:12.4f} {R2_MLP_DE_SHUF:10.4f}")
print(f"CNN                    {R2_CNN_DE:12.4f} {R2_CNN_DE_SHUF:10.4f}")

fig, ax = plt.subplots(figsize=(6.6, 4))
x = np.arange(2); w = 0.36
ax.bar(x - w/2, [R2_MLP_DE, R2_CNN_DE], w, color=C_DATA, label="real microstructures")
ax.bar(x + w/2, [R2_MLP_DE_SHUF, R2_CNN_DE_SHUF], w, color=C_BAD, label="pixels shuffled")
ax.set_xticks(x); ax.set_xticklabels(["flattened dense network", "CNN"])
ax.axhline(0, color="k", lw=0.8)
lo_ = min(0, R2_MLP_DE, R2_MLP_DE_SHUF, R2_CNN_DE_SHUF) - 0.1
ax.set_ylim(lo_, 1.0); ax.set_ylabel("test $R^2$ on $\\Delta E$")
ax.legend(fontsize=8, loc="upper left")
for xv, v in zip([-w/2, 1 - w/2, w/2, 1 + w/2], [R2_MLP_DE, R2_CNN_DE, R2_MLP_DE_SHUF, R2_CNN_DE_SHUF]):
    ax.text(xv, max(v, 0) + 0.015, f"{v:.3f}", ha="center", fontsize=9)
ax.set_title("the shuffle test on the anisotropy", fontsize=10)
plt.tight_layout(); plt.show()


**What the table and the bars show.** The CNN's score collapses to about zero when the pixels are
shuffled. The dense network's two scores are close to each other, as Notebook 3a said they must be,
since a fixed shuffle is invisible to a dense layer. They are also both close to zero: even on the
real images, the flattened network found almost nothing of the anisotropy. It never had a way to
use the arrangement, so shuffling had nothing to take away.

Together this is good evidence that the CNN uses the arrangement of the fibres, which is what
$\Delta E$ depends on.

---

# Part 5 - How far does a unit need to see?

Notebook 3a showed that each unit in a CNN sees only a square patch of the input, its receptive
field, and that the patch grows with every block. The descriptor that carries the anisotropy
compares pixels a fixed distance apart, the lag printed in Part 3. This raises a question: **does
the network need to see that far?**

The experiment: four networks with 1, 2, 3 and 4 blocks of conv, ReLU and pool. Each ends with a
$1 \times 1$ convolution and **global average pooling**, which averages each channel over the whole
map, so that the only thing that changes between the four is how far one unit can see. Everything
else is the same: data, split, augmentation, number of epochs.

In [ ]:
# --- Receptive-field ladder on dE ---------------------------------------------------
def make_gap_cnn(n_blocks, widths=(16, 32, 32, 32), seed=SEED):
    torch.manual_seed(seed)
    layers, c = [], 1
    for w in widths[:n_blocks]:
        layers += [nn.Conv2d(c, w, 3, padding=1, padding_mode="circular"), nn.ReLU(), nn.MaxPool2d(2)]
        c = w
    layers += [nn.Conv2d(c, 32, 1), nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32, 1)]
    return nn.Sequential(*layers)

def receptive_field(n_blocks):
    r, J = 1, 1
    for _ in range(n_blocks):
        r, J = r + 2 * J, J          # 3x3 convolution, stride 1
        r, J = r + J, 2 * J          # 2x2 pool, stride 2
    return r

RF_LADDER = []
t_all = time.time()
for nb_ in [1, 2, 3, 4]:
    m_ = make_gap_cnn(nb_)
    npar_ = sum(p.numel() for p in m_.parameters())
    res_ = train_cnn(m_, Xtr_img, yd_tr, Xte_img, yd_te, epochs=EPOCHS_RF, lr=2e-3,
                     tag=f"{nb_} block(s), field {receptive_field(nb_)} px, ")
    RF_LADDER.append((nb_, receptive_field(nb_), npar_, res_["r2"]))
print(f"\nall four: {time.time() - t_all:.0f} s\n")
print(f"{'blocks':>6s} {'field (px)':>11s} {'weights':>8s} {'test R2 on dE':>14s}")
for nb_, rf_, np_, r2_ in RF_LADDER:
    print(f"{nb_:>6d} {rf_:>11d} {np_:>8,d} {r2_:>14.4f}")
print(f"\nlag of the dominant descriptor {TOP_DESC}: {LAG_TOP} px")

# typical fibre size in pixels, from connected white regions in a sample of images
from scipy import ndimage
diams = []
for im in X_IMG[::30]:
    lab, n = ndimage.label(im)
    areas = ndimage.sum(im, lab, range(1, n + 1))
    diams += list(2 * np.sqrt(np.asarray(areas) / np.pi))
FIBRE_D = float(np.median(diams))
print(f"median fibre diameter: about {FIBRE_D:.0f} px (fibres cut by the cell edge or touching each other blur this)")


### Animation: receptive field and score

**Left:** a microstructure with the receptive field of one unit drawn as an orange square, for each
of the four networks in turn. The green bar is the lag of the dominant descriptor, for scale.
**Right:** the test $R^2$ of each network, added as its square is drawn.

In [ ]:
# --- Animation 13: receptive field against score -----------------------------------
img13 = X_IMG[int(np.argmax(y_dE))]
fig, (axI, axB) = plt.subplots(1, 2, figsize=(12, 5), dpi=80, gridspec_kw={"width_ratios": [1, 1.2]})
axI.imshow(img13, cmap="gray", interpolation="nearest")
axI.set_xticks([]); axI.set_yticks([]); axI.grid(False)
sq = Rectangle((0, 0), 1, 1, fc=C_FIT, alpha=0.35, ec=C_FIT, lw=2.5)
axI.add_patch(sq)
axI.plot([4, 4 + LAG_TOP], [60, 60], color=C_ALT, lw=6, solid_capstyle="butt")
axI.text(4, 56.5, f"descriptor lag, {LAG_TOP} px", color=C_ALT, fontsize=10, weight="bold",
         bbox=dict(fc="w", ec="none", alpha=0.85))
labels13 = [f"{nb_} block{'s' if nb_ > 1 else ''}\nfield {rf_} px" for nb_, rf_, _, _ in RF_LADDER]
bars13 = axB.bar(range(4), [0, 0, 0, 0], color=C_DATA, width=0.6)
axB.axhline(R2_DESC_DE, color=C_ALT, ls="--", lw=1.8, label=f"14 descriptors, {R2_DESC_DE:.3f}")
axB.axhline(0, color="k", lw=0.8)
axB.set_xticks(range(4)); axB.set_xticklabels(labels13)
lo13 = min(-0.1, min(r[3] for r in RF_LADDER) - 0.05)
axB.set_ylim(lo13, 1.0); axB.set_ylabel("test $R^2$ on $\\Delta E$"); axB.legend(fontsize=9, loc="upper left")
vals13 = [axB.text(k, 0, "", ha="center", va="bottom", fontsize=11) for k in range(4)]
FR13 = [k for k in range(4) for _ in range(4)] + [3] * 3

def frame_rf13(f):
    k = FR13[f]
    nb_, rf_, _, r2_ = RF_LADDER[k]
    c0 = 32 - rf_ / 2
    sq.set_bounds(c0 - 0.5, c0 - 0.5, min(rf_, 64), min(rf_, 64))
    axI.set_title(f"{nb_} block{'s' if nb_ > 1 else ''}: one unit sees {rf_} x {rf_} pixels", fontsize=11)
    for j, b_ in enumerate(bars13):
        b_.set_height(RF_LADDER[j][3] if j <= k else 0)
        b_.set_color(C_FIT if j == k else C_DATA)
        vals13[j].set_text(f"{RF_LADDER[j][3]:.3f}" if j <= k else "")
        vals13[j].set_y(max(RF_LADDER[j][3], 0) + 0.01)
    return []

anim = animation.FuncAnimation(fig, frame_rf13, frames=len(FR13), interval=450)
plt.close(fig)
HTML(anim.to_jshtml())


**What the animation shows.** Read the scores in the printed table. With one block, a unit sees a
4x4 patch, smaller than a typical fibre (the printed median diameter), and the network cannot predict
the anisotropy at all. With two blocks the field reaches the descriptor lag (the green bar), and the
network recovers a large part of it. Beyond that, extra blocks change the score by much less than
that first step did, and not always upwards.

The reason is physical. Anisotropy is a property of how fibres sit
relative to their neighbours, several pixels away. A network that only sees a pixel's immediate
surroundings cannot measure it, however well it is trained. When you design a CNN for a mechanics
problem, ask what length scale the answer depends on, and make sure the receptive field covers it.

---

# Part 6 - Every method side by side

One more representation for the comparison: the two-point correlation map $S_2$ of Notebook 2,
compressed to 50 numbers with PCA, then gradient boosting.

In [ ]:
# --- Two-point correlation + PCA-50, the 2019 route, on this split -----------
t0 = time.time()

def s2_map(im):
    v = im.astype(np.float64)
    f = np.fft.rfft2(v)
    return np.fft.fftshift(np.fft.irfft2(f * np.conj(f), s=v.shape) / v.size)

S2_FLAT = np.array([s2_map(im).ravel() for im in X_IMG])
el_s2 = time.time() - t0

from sklearn.decomposition import PCA
pca_s2 = PCA(n_components=50, svd_solver="full", random_state=SEED).fit(S2_FLAT[itr])
S2_PC = pca_s2.transform(S2_FLAT)

gb_s2_dE = HistGradientBoostingRegressor(random_state=SEED, max_iter=400).fit(S2_PC[itr], yd_tr)
R2_S2_DE = r2_score(yd_te, gb_s2_dE.predict(S2_PC[ite]))
gb_s2_m  = HistGradientBoostingRegressor(random_state=SEED, max_iter=400).fit(S2_PC[itr], ym_tr)
R2_S2_MEAN = r2_score(ym_te, gb_s2_m.predict(S2_PC[ite]))

# composition alone: volume fraction and diameter, same model
X_COMP = clean[["vol_frac", "diameter"]].values.astype(np.float32)
gb_c_dE = HistGradientBoostingRegressor(random_state=SEED, max_iter=400).fit(X_COMP[itr], yd_tr)
R2_COMP_DE = r2_score(yd_te, gb_c_dE.predict(X_COMP[ite]))
gb_c_m  = HistGradientBoostingRegressor(random_state=SEED, max_iter=400).fit(X_COMP[itr], ym_tr)
R2_COMP_MEAN = r2_score(ym_te, gb_c_m.predict(X_COMP[ite]))

print(f"S2 for {len(X_IMG)} images: {el_s2:.1f} s, one map is {S2_FLAT.shape[1]} numbers")
print(f"PCA-50 keeps {pca_s2.explained_variance_ratio_.sum():.4f} of the variance of S2")
print(f"S2 + PCA-50      test R2   E_mean {R2_S2_MEAN:.4f}   dE {R2_S2_DE:.4f}")
print(f"composition      test R2   E_mean {R2_COMP_MEAN:.4f}   dE {R2_COMP_DE:.4f}")


In [ ]:
# --- The summary figure for the first day ------------------------------------
LADDER = [
    ("composition\n$V_f$ and fibre diameter", "composition, Vf and diameter",
     2, R2_COMP_MEAN, R2_COMP_DE, "#8C8C8C"),
    ("two-point correlation $S_2$\ncompressed to 50 PCs", "S2 + PCA-50",
     50, R2_S2_MEAN, R2_S2_DE, C_DATA),
    ("14 named descriptors\nchosen for the physics", "14 named descriptors",
     14, R2_DESC_MEAN, R2_DESC_DE, C_ALT),
    ("full descriptor bank", "full descriptor bank",
     30, R2_D30_MEAN, R2_D30_DE, "#2F6B43"),
    ("raw image 64x64\nCNN trained in this notebook", "raw image, CNN in this notebook",
     4096, R2_CNN_MEAN, R2_CNN_DE, C_FIT),
]
labels = [f"{r[0]}  ({r[2]})" for r in LADDER]
ypos = np.arange(len(LADDER))[::-1]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13.2, 5.0), sharey=True)
for ax, col, title in [(a1, 3, "$E_{mean}$, average stiffness"),
                       (a2, 4, "$\\Delta E$, anisotropy")]:
    vals = [r[col] for r in LADDER]
    ax.barh(ypos, vals, color=[r[5] for r in LADDER], height=0.62)
    for yv, v in zip(ypos, vals):
        ax.text(v + 0.02 if v > 0 else 0.02, yv, f"{v:.3f}", va="center", fontsize=9.5)
    ax.axvline(0, color="k", lw=0.9)
    ax.set_xlim(min(-0.15, min(vals) - 0.06), 1.22)
    ax.set_xlabel("test $R^2$"); ax.set_title(title, fontsize=11)
    ax.grid(axis="y", alpha=0)

a2.axvline(R2_PUB_DE, color=C_BAD, ls=":", lw=2.0)
a2.text(R2_PUB_DE - 0.03, ypos[0] + 0.62, f"tuned research CNN\npublished, {R2_PUB_DE:.3f}",
        fontsize=8.5, color=C_BAD, ha="right", va="top")
a2.set_ylim(ypos[-1] - 0.6, ypos[0] + 1.1)
a1.set_yticks(ypos); a1.set_yticklabels(labels, fontsize=9.5)
plt.suptitle("what the input representation is worth, same split, same 64x64 images\n"
             "bracketed number is the dimension of the representation", y=1.03, fontsize=11)
plt.tight_layout(); plt.show()

print(f"{'input representation':34s} {'dim':>5s} {'E_mean':>9s} {'dE':>9s}")
for _, nm, d_, a, b, _c in LADDER:
    print(f"{nm:34s} {d_:5d} {a:9.4f} {b:9.4f}")
print(f"{'published tuned CNN (reference)':34s} {'-':>5s} {'-':>9s} {R2_PUB_DE:9.4f}")


**What the figure shows.** Each bar is one way of describing the same 64x64 images, and the
bracketed number is how many numbers go into the model.

On the left, every method does well, because the average stiffness depends mainly on how much fibre
there is. Looking only at this panel, you would conclude the representation does not matter.

On the right the choice matters a great deal. Composition alone carries none of the anisotropy. The
two-point correlation recovers a good part of it. The named descriptors do better with fewer numbers,
and the full set of 30 does better again. The CNN sits below the descriptor bars. The dotted line is
the published, tuned CNN, which shows what a CNN can reach with a designed architecture and a much
larger training budget.

There is also a cost side. The descriptor route needs a person to design descriptors once, then costs
very little to fit. The CNN route needs no feature design, but costs minutes here and hours at
research scale.

---

# Part 7 - What did the network learn?

The network trained on $\Delta E$ in Part 3 is used below. Three views follow, and each one is
easier to over-read than the one before.

### The first-layer kernels

Sixteen 3x3 kernels, learned rather than written down. Read them as in Notebook 3a: the sum of the
weights says whether a kernel responds to level or only to change, and the pattern of signs says
which direction of change it prefers.

In [ ]:
# --- First layer filters ------------------------------------------------------
model_dE = res_dE["model"].cpu()
W1 = model_dE[0].weight.detach().numpy()[:, 0]          # (16, 3, 3)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
m = np.abs(W1).max()
for k, ax in enumerate(axes.ravel()):
    ax.imshow(W1[k], cmap="coolwarm", vmin=-m, vmax=m, interpolation="nearest")
    ax.set_title(f"sum {W1[k].sum():+.2f}", fontsize=8)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.suptitle("learned first layer filters", y=1.02, fontsize=11)
plt.tight_layout(); plt.show()

# how directional is each filter: horizontal minus vertical difference energy
dx = np.abs(np.diff(W1, axis=2)).sum(axis=(1, 2))
dy = np.abs(np.diff(W1, axis=1)).sum(axis=(1, 2))
print(f"{'filter':>7s} {'sum':>8s} {'x-variation':>12s} {'y-variation':>12s}")
for k in np.argsort(-(np.abs(dx - dy)))[:6]:
    print(f"{k:>7d} {W1[k].sum():>8.3f} {dx[k]:>12.3f} {dy[k]:>12.3f}")
print("\nFilters with x-variation far from y-variation are directional, which is what a target")
print("built from the difference between two directions would be expected to need.")


### Feature maps

One test microstructure, the most anisotropic in the test set, passed through the trained network.
The six most active channels are shown after each block. Early maps still look like the
microstructure with the boundaries picked out. Later maps are coarse, and no single channel needs to
have a name.

In [ ]:
# --- Feature maps through the trained network --------------------------------
probe_i = int(np.argmax(np.abs(yd_te)))                 # the most anisotropic test case
probe_img = Xte_img[probe_i]
probe_t = torch.tensor(probe_img).view(1, 1, 64, 64)

acts, h = [], probe_t
for layer in model_dE:
    h = layer(h)
    if isinstance(layer, nn.MaxPool2d):
        acts.append(h.detach().numpy()[0])

fig, axes = plt.subplots(3, 7, figsize=(14, 6.4))
for r, maps in enumerate(acts):
    axes[r, 0].imshow(probe_img, cmap="gray", interpolation="nearest")
    axes[r, 0].set_ylabel(f"block {r+1}\n{maps.shape}", fontsize=8)
    order = np.argsort(-maps.std(axis=(1, 2)))[:6]      # the six most active channels
    for c, k in enumerate(order):
        axes[r, c+1].imshow(maps[k], cmap="viridis", interpolation="nearest")
        axes[r, c+1].set_title(f"ch {k}", fontsize=8)
for a in axes.ravel():
    a.set_xticks([]); a.set_yticks([]); a.grid(False)
plt.suptitle(f"feature maps for one test microstructure, "
             f"$\\Delta E$ = {yd_te[probe_i]:+.2f} GPa", y=1.0, fontsize=11)
plt.tight_layout(); plt.show()


### Try it: any test cell, any block

Pick a test microstructure and a block. The six most active channels after that block are shown,
with the network's prediction and the true value.

In [ ]:
# --- Interactive: feature maps for any test cell -----------------------------------
def fmap_explorer(cell=0, block=1):
    img = Xte_img[cell]
    h = torch.tensor(img).view(1, 1, 64, 64)
    seen = 0
    with torch.no_grad():
        for layer in model_dE:
            h = layer(h)
            if isinstance(layer, nn.MaxPool2d):
                seen += 1
                if seen == block:
                    maps = h[0].numpy()
        pred_ = float(model_dE(torch.tensor(img).view(1, 1, 64, 64)))
    order = np.argsort(-maps.std(axis=(1, 2)))[:6]
    fig, axes = plt.subplots(1, 7, figsize=(14, 2.6))
    axes[0].imshow(img, cmap="gray", interpolation="nearest")
    axes[0].set_title(f"true {yd_te[cell]:+.2f}, predicted {pred_:+.2f} GPa", fontsize=8)
    for ax, k in zip(axes[1:], order):
        ax.imshow(maps[k], cmap="viridis", interpolation="nearest")
        ax.set_title(f"block {block}, channel {k}\n{maps.shape[1]}x{maps.shape[2]}", fontsize=8)
    for a in axes:
        a.set_xticks([]); a.set_yticks([]); a.grid(False)
    plt.tight_layout(); plt.show()

interact(fmap_explorer, cell=IntSlider(value=0, min=0, max=len(Xte_img) - 1, step=1, continuous_update=False),
         block=IntSlider(value=1, min=1, max=3, step=1, continuous_update=False));


### The same forward pass, animated

The animation walks the same microstructure through every stage of the trained network. **Left:**
every channel at the current stage, each scaled to its own range. **Right:** the side length of the
maps and the number of channels at each stage. The tiles get smaller and more numerous, then turn
into 64 numbers and finally one prediction.

In [ ]:
# --- Animation: one microstructure through the trained network ---------------
# Same trained model and same microstructure as the figure above.
hcur = torch.tensor(probe_img).view(1, 1, 64, 64)
SP_NAMES = ["conv 1 + ReLU", "pool 1", "conv 2 + ReLU", "pool 2", "conv 3 + ReLU", "pool 3"]
SP = [("input", probe_img[None].astype(np.float32))]
ks = 0
dense_act = None
for layer in model_dE:
    hcur = layer(hcur)
    if hcur.dim() == 4 and isinstance(layer, (nn.ReLU, nn.MaxPool2d)):
        SP.append((SP_NAMES[ks], hcur.detach().numpy()[0])); ks += 1
    if hcur.dim() == 2 and isinstance(layer, nn.ReLU):
        dense_act = hcur.detach().numpy()[0]
pred_probe = float(hcur.detach().numpy().ravel()[0])

SNAME = [s[0] for s in SP] + ["dense 64 + ReLU", "output"]
CH    = [1, 16, 16, 32, 32, 64, 64, 64, 1]
SPAT  = [64, 64, 32, 32, 16, 16, 8, 1, 1]
NVAL  = [c * s * s for c, s in zip(CH, SPAT)]
N_ST  = len(SNAME)                      # 9 stages
FPS_H = 6
N_FR_H = N_ST * FPS_H                   # 54 frames

print(f"{'stage':18s} {'channels':>9s} {'size':>6s} {'values':>9s}")
for nm, c, s, v in zip(SNAME, CH, SPAT, NVAL):
    print(f"{nm:18s} {c:>9d} {s:>6d} {v:>9,d}")
print(f"\nprediction for this microstructure  {pred_probe:+.3f} GPa")
print(f"true value                          {yd_te[probe_i]:+.3f} GPa")

GAP = 2
def tile_layout(n, S):
    cols = 1 if n == 1 else (4 if n <= 16 else 8)
    rows = int(np.ceil(n / cols))
    return rows, cols, cols * S + (cols - 1) * GAP, rows * S + (rows - 1) * GAP

cmap_v = plt.cm.viridis
cmap_g = plt.cm.gray

def mosaic(a, nshow, cm):
    # Returns an RGBA mosaic. Channels not yet revealed are drawn light grey, so a
    # partly filled stage never reads as an empty box.
    n, S, _ = a.shape
    rows, cols, W, H = tile_layout(n, S)
    rgba = np.ones((H, W, 4), np.float32)
    order = np.argsort(-a.std(axis=(1, 2))) if n > 1 else np.arange(n)
    for t in range(n):
        ch = order[t]; r, c = t // cols, t % cols
        if t < nshow:
            tile = a[ch].astype(np.float32)
            lo, hi = float(tile.min()), float(tile.max())
            tile = (tile - lo) / (hi - lo) if hi > lo else np.zeros_like(tile)
            px = cm(tile)
        else:
            px = np.ones((S, S, 4), np.float32); px[..., :3] = 0.88
        rgba[r*(S+GAP):r*(S+GAP)+S, c*(S+GAP):c*(S+GAP)+S] = px
    return rgba, W, H

fig = plt.figure(figsize=(13.4, 5.6))
gsh = fig.add_gridspec(1, 2, width_ratios=[1.35, 1.0], wspace=0.22)
axM, axC = fig.add_subplot(gsh[0, 0]), fig.add_subplot(gsh[0, 1])
axM.grid(False); axM.set_xticks([]); axM.set_yticks([])
for sp in ("top", "right", "bottom", "left"):
    axM.spines[sp].set_visible(False)
axM.set_xlim(-8, 288); axM.set_ylim(300, -40)

im_map = axM.imshow(np.ones((2, 2, 4), np.float32), extent=(0, 64, 64, 0),
                    interpolation="nearest", zorder=2)
im_vec = axM.imshow(np.full((1, 64), np.nan), cmap=plt.cm.viridis, extent=(0, 256, 138, 122),
                    interpolation="nearest", zorder=2)
im_vec.set_visible(False)
frame_r = Rectangle((0, 0), 64, 64, fill=False, ec=C_FIT, lw=1.6, zorder=4)
axM.add_patch(frame_r)
out_box = Rectangle((0, 102), 56, 56, fc="#f2e4c9", ec=C_FIT, lw=1.8, zorder=3, visible=False)
axM.add_patch(out_box)
out_txt = axM.text(28, 130, "", ha="center", va="center", fontsize=13, zorder=5)
cap_h   = axM.text(0.0, 1.03, "", ha="left", va="bottom", fontsize=11.5, transform=axM.transAxes)
sub_h   = axM.text(0.0, -0.03, "", ha="left", va="top", fontsize=10, color="0.35", transform=axM.transAxes)

axC.set_yscale("log", base=2)
axC.set_xlim(-0.4, N_ST - 0.6); axC.set_ylim(0.55, 200)
axC.set_xticks(range(N_ST)); axC.set_xticklabels(SNAME, rotation=38, ha="right", fontsize=8)
axC.set_ylabel("count")
ln_s, = axC.plot([], [], "-o", color=C_DATA, lw=2, ms=5, drawstyle="steps-post",
                 label="spatial size, pixels per side")
ln_c, = axC.plot([], [], "-s", color=C_FIT, lw=2, ms=5, drawstyle="steps-post",
                 label="channels")
mk_s, = axC.plot([], [], "o", ms=11, mfc="none", mec=C_DATA, mew=2)
mk_c, = axC.plot([], [], "s", ms=11, mfc="none", mec=C_FIT, mew=2)
axC.legend(fontsize=8.5, loc="lower right")
axC.set_title("resolution falls, channel count rises", fontsize=10)
info_h = axC.text(0.98, 0.97, "", transform=axC.transAxes, ha="right", va="top", fontsize=9,
                  family="monospace", bbox=dict(fc="w", ec="0.75", boxstyle="round,pad=0.35"))

def update_hier(f):
    s, t = f // FPS_H, (f % FPS_H) / (FPS_H - 1)
    im_map.set_visible(False); im_vec.set_visible(False); out_box.set_visible(False)
    out_txt.set_text(""); frame_r.set_visible(True)
    if s < 7:
        a = SP[s][1]
        n = a.shape[0]
        nshow = max(1, int(np.ceil(t * n))) if n > 1 else 1
        M, W, H = mosaic(a, nshow, cmap_g if s == 0 else cmap_v)
        y0 = 130.0 - H / 2.0
        im_map.set_data(M); im_map.set_extent((0, W, y0 + H, y0))
        im_map.set_visible(True)
        frame_r.set_xy((0, y0)); frame_r.set_width(W); frame_r.set_height(H)
        axM.set_xlim(-3, W + 3); axM.set_ylim(y0 + H + 3, y0 - 3)
        sub_h.set_text(f"{nshow} of {n} channel" + ("s" if n > 1 else "") +
                       f" drawn, each {SPAT[s]} x {SPAT[s]} pixels")
        cap_h.set_text(f"stage {s+1} of {N_ST}:  {SNAME[s]}   ({CH[s]} x {SPAT[s]} x {SPAT[s]})")
    elif s == 7:
        nshow = max(1, int(np.ceil(t * 64)))
        v = dense_act.astype(np.float32).copy(); v[nshow:] = np.nan
        im_vec.set_data(v.reshape(1, 64))
        im_vec.set_clim(0, max(float(dense_act.max()), 1e-6)); im_vec.set_visible(True)
        frame_r.set_xy((0, 122)); frame_r.set_width(256); frame_r.set_height(16)
        axM.set_xlim(-6, 262); axM.set_ylim(150, 110)
        sub_h.set_text("the 4096 values are flattened and mixed by a dense layer.\n"
                       "no spatial structure is left, only 64 numbers")
        cap_h.set_text(f"stage {s+1} of {N_ST}:  {SNAME[s]}")
    else:
        out_box.set_visible(True); frame_r.set_visible(False)
        axM.set_xlim(-40, 96); axM.set_ylim(180, 80)
        out_txt.set_text(f"$\\Delta E$\n{pred_probe:+.2f}")
        sub_h.set_text(f"one number. true $\\Delta E$ = {yd_te[probe_i]:+.2f} GPa")
        cap_h.set_text(f"stage {s+1} of {N_ST}:  {SNAME[s]}")
    xs_ = np.arange(s + 1)
    ln_s.set_data(xs_, SPAT[:s+1]); ln_c.set_data(xs_, CH[:s+1])
    mk_s.set_data([s], [SPAT[s]]);  mk_c.set_data([s], [CH[s]])
    info_h.set_text(f"channels {CH[s]:>7d}\nsize     {SPAT[s]:>7d}\nvalues   {NVAL[s]:>7,d}")
    return []

fig.subplots_adjust(left=0.02, right=0.97, bottom=0.30, top=0.90)
anim_hier = animation.FuncAnimation(fig, update_hier, frames=N_FR_H, interval=350, blit=False)
plt.close(fig)
HTML(anim_hier.to_jshtml())


**What the animation shows.** After the first block the fibre boundaries are visible in most
channels. After the third block each map is 8 by 8, one value for each 8 by 8 pixel block of the
cell, and no channel looks like a picture of anything. At the dense layer the spatial layout is
gone. Whatever geometric information the network uses must have been extracted before that point.

### Saliency: which pixels matter most?

**Saliency** is the gradient of the prediction with respect to each input pixel, computed with one
backward pass, as in Notebook 3a Part 4. A bright pixel is one where a small change would move
the prediction most. Read the warnings after the figure before drawing conclusions.

In [ ]:
# --- Gradient saliency --------------------------------------------------------
def saliency(model, img, smooth=1.5):
    x = torch.tensor(img).view(1, 1, 64, 64).requires_grad_(True)
    model.zero_grad()
    out = model(x)
    out.backward()
    g = x.grad.detach().numpy()[0, 0]
    return ndimage.gaussian_filter(np.abs(g), smooth)

order_te = np.argsort(np.abs(yd_te))[::-1][:3]          # three strongly anisotropic cases
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for c, i in enumerate(order_te):
    img = Xte_img[i]
    s = saliency(model_dE, img)
    axes[0, c].imshow(img, cmap="gray", interpolation="nearest")
    axes[0, c].set_title(f"true $\\Delta E$ = {yd_te[i]:+.2f}\n"
                         f"predicted {res_dE['pred'][i]:+.2f} GPa", fontsize=9)
    axes[1, c].imshow(img, cmap="gray", interpolation="nearest", alpha=0.85)
    axes[1, c].imshow(s, cmap="inferno", alpha=0.55, interpolation="bilinear")
    axes[1, c].set_title("saliency, smoothed", fontsize=9)
for a in axes.ravel():
    a.set_xticks([]); a.set_yticks([]); a.grid(False)
plt.tight_layout(); plt.show()

# how much of the sensitivity sits on fibre boundaries rather than in the bulk
img0 = Xte_img[order_te[0]]
s0   = saliency(model_dE, img0)
edge = ndimage.binary_dilation(
    ndimage.laplace(img0) != 0, iterations=1)
print(f"fraction of pixels on or next to a fibre boundary: {edge.mean():.3f}")
print(f"fraction of total saliency there:                  {s0[edge].sum()/s0.sum():.3f}")


### How much to believe

The printed numbers compare the share of pixels on or next to a fibre boundary with the share of the
saliency that falls there. If the second is larger, the network's sensitivity is concentrated on
boundaries, which fits a model that responds to the arrangement of interfaces. This is weak
supporting evidence and no more.

1. **Saliency measures sensitivity, not cause.** It says the output would change if that pixel
   changed. It does not say the network used that region to reach its answer.
2. **It is local.** The gradient is taken at one image, and a binary image cannot really be changed
   by a small amount.
3. **A model that predicts badly still produces a convincing-looking saliency map.** This network
   leaves a good part of $\Delta E$ unexplained, and any reading of its internals inherits that.

---

# What to take away

1. On the average stiffness, the CNN works well, and so does a straight line in volume fraction.
   Pick a test problem where methods can actually differ before comparing them.
2. On the anisotropy, a quick CNN lost to fourteen descriptors chosen with physical insight, fitted
   in a fraction of the time. Always run the physics-based baseline.
3. The shuffle test showed that the CNN uses the arrangement of the fibres, and the dense network
   does not.
4. A unit can only use what lies inside its receptive field. When the field was smaller than a fibre,
   the anisotropy could not be predicted at all. Match the receptive field to the length scale of the
   physics.
5. Augmentation must respect the physics: periodic shifts and mirrors are safe here, a 90 degree
   rotation is not unless the target is transformed too.
6. Filters, feature maps and saliency are useful checks, not proof of what the network understands.

# Exercises

### 1. Read the weight count
From the printed network summary in Part 2, what fraction of the weights is in the convolution
layers? Use the formula $P_{\text{conv}} = k^2 c_{in} c_{out} + c_{out}$ to check the count for the
second convolution layer by hand.

### 2. Break the CNN on purpose
Retrain the CNN on $\Delta E$ with `augmented=False`, and again with `pad_mode="zeros"` in
`make_cnn`. Compare both scores with Part 3. Which change costs more?

### 3. The descriptors were better
Make a small table of test $R^2$, fitting time and number of inputs for the 14 descriptors, the 30
descriptors and the CNN on $\Delta E$. If you had to predict $\Delta E$ for ten thousand new cells
tomorrow, which would you use, and why? What does the descriptor route cost that the table does not
show?

### 4. Rotation done properly
Add a 90 degree rotation to `augment`, and flip the sign of the $\Delta E$ target whenever it is
applied (you will need to change `train_cnn` so the target is transformed with the image). Does it
help? Then add the rotation without flipping the sign and report what happens.

### 5. A different way to see further
In Part 5, keep two blocks but replace the $3 \times 3$ kernels by $7 \times 7$ kernels (padding 3).
Work out the new receptive field, retrain, and compare the score and the number of weights with the
three-block network.

### 6. Replace the dense head
The network of Part 2 flattens the 64x8x8 output into a dense layer, which is where most of its
weights sit. Replace that head by global average pooling followed by one dense layer, as in Part 5,
retrain on $\Delta E$ for the same number of epochs, and compare score and weight count. Explain from
the physics why removing the dependence on absolute position might be reasonable for a periodic
cell.